In [ ]:
#task 1: inductive biases and feature representations

# drive holds the repo and the cached tensors, so nothing is lost when disconnecting
from google.colab import drive
drive.mount("/content/drive")
!pip install -q open_clip_torch

In [ ]:
import copy
import json
import math
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import open_clip
import pandas as pd
import sklearn
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from sklearn.manifold import TSNE
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
from torchvision.datasets import STL10
from torchvision.models import ResNet50_Weights, ViT_B_16_Weights, resnet50, vit_b_16

CFG = {
    # fixed by the assignment manual:
    "seed": 6304,
    "image_size": 224,
    "val_fraction": 0.2,            # 80/20 split of the official train set
    "eval_per_class": 50,           # 10 classes x 50 = 500 test images
    "clip_prompt": "a photo of a {}.",
    "head": {"max_epochs": 50, "lr": 1e-3, "weight_decay": 1e-4, "patience": 5,
             "batch_size": 64},     # batch size is not specified anywhere, so it is recorded here
    "translation_shifts": [0, 8, 16, 32],
    "patch_grid": 4,

    # my design choices
    "dataset": "stl10",
    "hue_degrees": 180,             # every hue mapped to its opposite, brightness kept
    "cue": {
        # five unordered pairs, every class used once, both directions are generated
        "pairs": [["cat", "car"], ["dog", "truck"], ["horse", "ship"],
                  ["deer", "airplane"], ["bird", "monkey"]],
        "alpha": 1.0,               # AdaIN style strength, 1 = full stylization
        "candidates_per_direction": 30,
        "min_total": 200,
        # written down before any model sees the images
        "rejection_rule": ("Reject a stylized image if (a) the object outline can not be identified by eye,  "
                           "(b) the image is mostly flat colour or the "
                           "texture is not visible, or (c) it looks the same as the content image."),
    },
    "projection": {"method": "tsne", "metric": "cosine", "perplexity": 30,
                   "n_neighbors": 15, "min_dist": 0.1,     # last two only matter for umap
                   "translation_shown": 32},               # which shift (px, to the right) goes in the plot
}

REPO = Path("/content/drive/MyDrive/atml-pa1")
RESULTS = REPO / "task1" / "results"          # small files, committed
FIGS = RESULTS / "figures"
CACHE = Path("/content/drive/MyDrive/pa1-data/task1")   # big tensors, never committed
DATA_ROOT = "/content/data"                   # dataset on the fast local disk
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(DEVICE)

In [ ]:
# first run only: put a copy of the repo on drive
if not REPO.exists():
    !git clone https://github.com/mardyweb/atml-pa1.git {REPO}

In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def load_json(path):
    with open(path) as f:
        return json.load(f)


def save_json(obj, path):
    with open(path, "w") as f:
        json.dump(obj, f, indent=1)


def save_table(df, name):
    """csv for the repo, tex to drop into the report, and show it here."""
    df.to_csv(RESULTS / f"{name}.csv", index=False)
    try:
        df.to_latex(RESULTS / f"{name}.tex", index=False, float_format="%.3f")
    except Exception:
        pass
    display(df.round(3))


for folder in [RESULTS, FIGS, RESULTS / "cue_conflict" / "sheets", CACHE, CACHE / "feats"]:
    folder.mkdir(parents=True, exist_ok=True)
set_seed(CFG["seed"])

# exact settings and library versions, kept next to the results
save_json({"config": CFG, "device": str(DEVICE),
           "versions": {"torch": torch.__version__, "torchvision": torchvision.__version__,
                        "open_clip": open_clip.__version__, "sklearn": sklearn.__version__,
                        "numpy": np.__version__}}, RESULTS / "run_info.json")

In [ ]:
# ---- colors for figures ----
MODEL_COLORS = {"resnet50_head": "#D1495B", "vit_b16_head": "#00798C",
                "clip_head": "#EDAE49", "clip_zeroshot": "#6A4C93"}
MODEL_LABELS = {"resnet50_head": "ResNet-50 + head", "vit_b16_head": "ViT-B/16 + head",
                "clip_head": "CLIP + head", "clip_zeroshot": "CLIP zero-shot"}
MODEL_MARKERS = {"resnet50_head": "o", "vit_b16_head": "s", "clip_head": "D", "clip_zeroshot": "^"}
BACKBONE_COLORS = {"resnet50": "#D1495B", "vit_b16": "#00798C", "clip": "#EDAE49"}
BACKBONE_LABELS = {"resnet50": "ResNet-50", "vit_b16": "ViT-B/16", "clip": "CLIP ViT-B/32"}
DECISION_COLORS = {"shape": "#33658A", "texture": "#F26419", "other": "#CFCABD"}
CLASS_COLORS = ["#D1495B", "#00798C", "#EDAE49", "#6A4C93", "#8AB17D",
                "#2E4057", "#F08A4B", "#B5838D", "#3FA7D6", "#5B3A29"]
INK = "#2B2B2B"
MODEL_TEXT_COLORS = {**MODEL_COLORS, "clip_head": "#C98A1B"}
STABILITY_CMAP = LinearSegmentedColormap.from_list(
    "stability", ["#FBF6EC", "#BFE0DA", "#4FA6A6", "#00798C", "#003F4A"])

plt.rcParams.update({
    "font.family": "DejaVu Sans", "font.size": 9, "axes.titlesize": 10, "axes.labelsize": 9,
    "axes.edgecolor": INK, "axes.labelcolor": INK, "xtick.color": INK, "ytick.color": INK,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "axes.axisbelow": True, "grid.color": "#E4E0D6", "grid.linewidth": 0.7,
    "legend.frameon": False, "legend.fontsize": 8,
    "figure.dpi": 110, "savefig.bbox": "tight", "pdf.fonttype": 42,
})


def save_fig(fig, name):
    """pdf for latex, png for a quick look."""
    fig.savefig(FIGS / f"{name}.pdf")
    fig.savefig(FIGS / f"{name}.png", dpi=300)
    plt.show()
    plt.close(fig)

## Data: fixed splits and the common 224x224 images

In [ ]:
def load_stl10(split):
    """Raw images (N,3,96,96 uint8), labels and class names."""
    ds = STL10(root=DATA_ROOT, split=split, download=True)
    return ds.data, np.asarray(ds.labels, dtype=np.int64), list(ds.classes)


def resize_to_common(data, chunk=500):
    """Upsample to 224x224 (bicubic) and keep uint8 (every intervention starts here)"""
    size, out = CFG["image_size"], []
    for start in range(0, len(data), chunk):
        x = torch.from_numpy(data[start:start + chunk]).float()
        x = F.interpolate(x, size=(size, size), mode="bicubic", align_corners=False)
        out.append(x.round().clamp(0, 255).to(torch.uint8))
    return torch.cat(out)


def make_splits(train_labels, test_labels, class_names):
    seed = CFG["seed"]
    # 80/20 split of the official train set, same class ratio in both parts
    train_idx, val_idx = train_test_split(np.arange(len(train_labels)), test_size=CFG["val_fraction"],
                                          stratify=train_labels, random_state=seed)
    # same number of test images from every class
    rng = np.random.default_rng(seed)
    eval_idx, shortfall = [], {}
    for c, name in enumerate(class_names):
        pool = np.where(test_labels == c)[0]
        k = min(CFG["eval_per_class"], len(pool))
        if k < CFG["eval_per_class"]:
            shortfall[name] = int(k)        # class too small, keep everything it has
        eval_idx.extend(sorted(rng.choice(pool, size=k, replace=False).tolist()))
    return {"dataset": CFG["dataset"], "seed": seed, "class_names": class_names,
            "train_idx": sorted(train_idx.tolist()), "val_idx": sorted(val_idx.tolist()),
            "eval_idx": eval_idx,                       # positions in the official test split
            "eval_labels": test_labels[eval_idx].tolist(),
            "eval_class_shortfall": shortfall}          # empty = perfectly balanced


if (RESULTS / "splits.json").exists() and (CACHE / "eval_clean.pt").exists():
    # already built in an earlier session
    SPLITS = load_json(RESULTS / "splits.json")
    CLEAN = torch.load(CACHE / "eval_clean.pt")
else:
    _, train_y, class_names = load_stl10("train")
    test_raw, test_y, _ = load_stl10("test")
    SPLITS = make_splits(train_y, test_y, class_names)
    save_json(SPLITS, RESULTS / "splits.json")
    CLEAN = resize_to_common(test_raw[SPLITS["eval_idx"]])
    torch.save(CLEAN, CACHE / "eval_clean.pt")
    del test_raw

CLASS_NAMES = SPLITS["class_names"]
LABELS = np.array(SPLITS["eval_labels"])
print(f"train {len(SPLITS['train_idx'])} | val {len(SPLITS['val_idx'])} | eval subset {len(SPLITS['eval_idx'])}")
print("eval images per class:", np.bincount(LABELS).tolist(), "| shortfall:", SPLITS["eval_class_shortfall"])

## Interventions
All of them take and return uint8 batches, so every model reads exactly the same pixels.

In [ ]:
# RGB <-> YIQ. Y is brightness, I and Q carry the colour.
RGB2YIQ = torch.tensor([[0.299, 0.587, 0.114],
                        [0.5959, -0.2746, -0.3213],
                        [0.2115, -0.5227, 0.3112]])


def to_uint8(x):
    return (x.clamp(0, 1) * 255).round().to(torch.uint8)


def grayscale(imgs):
    """Drop colour, keep brightness (copied into all three channels)."""
    x = imgs.float() / 255
    luma = (x * RGB2YIQ[0].view(1, 3, 1, 1)).sum(dim=1, keepdim=True)
    return to_uint8(luma.expand(-1, 3, -1, -1))


def hue_rotate(imgs, degrees):
    """Rotate every colour by a fixed angle without touching brightness.
    The (I,Q) colour vector is rotated, Y stays as it is. Shapes, edges and
    brightness are unchanged, only the hues are wrong."""
    t = math.radians(degrees)
    rot = torch.tensor([[1.0, 0.0, 0.0],
                        [0.0, math.cos(t), -math.sin(t)],
                        [0.0, math.sin(t), math.cos(t)]])
    full = torch.linalg.inv(RGB2YIQ) @ rot @ RGB2YIQ     # one 3x3 matrix, rgb -> rgb
    return to_uint8(torch.einsum("ij,njhw->nihw", full, imgs.float() / 255))


def translate(imgs, dx=0, dy=0):
    """Shift the content by (dx, dy) pixels, dx>0 is right and dy>0 is down.
    Reflection-pad first, then crop back to the original size at a shifted
    position, so no black borders appear."""
    pad = max(abs(dx), abs(dy))
    if pad == 0:
        return imgs.clone()
    h, w = imgs.shape[-2:]
    padded = F.pad(imgs.float(), (pad, pad, pad, pad), mode="reflect")
    top, left = pad - dy, pad - dx
    return padded[..., top:top + h, left:left + w].to(torch.uint8)


DIRECTIONS = {"right": (1, 0), "left": (-1, 0), "down": (0, 1), "up": (0, -1)}


def apply_patch_perm(img, perm, grid):
    """Reorder the grid x grid patches of one image."""
    c, h, w = img.shape
    ph, pw = h // grid, w // grid
    patches = img.unfold(1, ph, ph).unfold(2, pw, pw)               # (C, grid, grid, ph, pw)
    patches = patches.reshape(c, grid * grid, ph, pw)[:, perm]      # new order
    patches = patches.reshape(c, grid, grid, ph, pw).permute(0, 1, 3, 2, 4)
    return patches.reshape(c, h, w)                                 # stitched back together


def patch_shuffle(imgs, grid, seed):
    """One random patch order per image. Returns the images and the orders used."""
    rng = np.random.default_rng(seed)
    identity = np.arange(grid * grid)
    out, perms = torch.empty_like(imgs), []
    for i in range(len(imgs)):
        perm = rng.permutation(grid * grid)
        while np.array_equal(perm, identity):       # redraw if nothing moved
            perm = rng.permutation(grid * grid)
        perms.append(perm.tolist())
        out[i] = apply_patch_perm(imgs[i], perm, grid)
    return out, perms

## Frozen backbones and linear heads

In [ ]:
BACKBONES = ["resnet50", "vit_b16", "clip"]
IMAGENET_MEAN, IMAGENET_STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)
CLIP_MEAN, CLIP_STD = (0.48145466, 0.4578275, 0.40821073), (0.26862954, 0.26130258, 0.27577711)


class FrozenBackbone(nn.Module):
    """uint8 images in, one feature vector per image out. Each model gets its own normalization."""

    def __init__(self, name):
        super().__init__()
        self.name = name
        if name == "resnet50":
            net = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)
            net.fc = nn.Identity()            # output = global-average-pooled feature (2048)
            mean, std = IMAGENET_MEAN, IMAGENET_STD
        elif name == "vit_b16":
            net = vit_b_16(weights=ViT_B_16_Weights.IMAGENET1K_V1)
            net.heads = nn.Identity()         # output = final class token (768)
            mean, std = IMAGENET_MEAN, IMAGENET_STD
        else:
            # the openai weights were trained with QuickGELU. Newer open_clip versions only
            # warn about this and use plain GELU, which hurts accuracy:
            net, _, _ = open_clip.create_model_and_transforms("ViT-B-32", pretrained="openai",
                                                              force_quick_gelu=True)
            mean, std = CLIP_MEAN, CLIP_STD
        self.net = net.eval()
        for p in self.net.parameters():
            p.requires_grad_(False)
        self.register_buffer("mean", torch.tensor(mean).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor(std).view(1, 3, 1, 1))

    @torch.no_grad()
    def forward(self, imgs_uint8):
        x = (imgs_uint8.float() / 255 - self.mean) / self.std
        if self.name == "clip":
            return F.normalize(self.net.encode_image(x), dim=-1)     # unit-length embedding
        return self.net(x)


_loaded = {}


def get_backbone(name):
    """Models are only loaded when something actually needs them."""
    if name not in _loaded:
        _loaded[name] = FrozenBackbone(name).to(DEVICE)
    return _loaded[name]


@torch.no_grad()
def extract_features(backbone, imgs_uint8, batch_size=100):
    feats = []
    for start in range(0, len(imgs_uint8), batch_size):
        feats.append(backbone(imgs_uint8[start:start + batch_size].to(DEVICE)).float().cpu())
    return torch.cat(feats)


def features_for(name, imgs):
    """Features of one image set from all three backbones, cached on drive.
    The same tensor goes to every model."""
    path = CACHE / "feats" / f"{name}.pt"
    if path.exists():
        return torch.load(path)
    out = {bb: extract_features(get_backbone(bb), imgs) for bb in BACKBONES}
    torch.save(out, path)
    return out

In [ ]:
def train_head(train_x, train_y, val_x, val_y, n_classes):
    """Linear layer on fixed features, early stopping on validation accuracy."""
    hc = CFG["head"]
    torch.manual_seed(CFG["seed"])                     # same init for every backbone
    head = nn.Linear(train_x.shape[1], n_classes).to(DEVICE)
    opt = torch.optim.AdamW(head.parameters(), lr=hc["lr"], weight_decay=hc["weight_decay"])
    loss_fn = nn.CrossEntropyLoss()
    train_x, train_y, val_x, val_y = [t.to(DEVICE) for t in (train_x, train_y, val_x, val_y)]
    shuffle_gen = torch.Generator().manual_seed(CFG["seed"])

    best_acc, best_state, bad_epochs, log = -1.0, None, 0, []
    for epoch in range(1, hc["max_epochs"] + 1):
        # one pass over the shuffled training features
        head.train()
        order = torch.randperm(len(train_x), generator=shuffle_gen).to(DEVICE)
        total = 0.0
        for start in range(0, len(order), hc["batch_size"]):
            idx = order[start:start + hc["batch_size"]]
            loss = loss_fn(head(train_x[idx]), train_y[idx])
            opt.zero_grad()
            loss.backward()
            opt.step()
            total += loss.item() * len(idx)

        # validation accuracy decides which epoch we keep
        head.eval()
        with torch.no_grad():
            val_acc = (head(val_x).argmax(1) == val_y).float().mean().item()
        log.append({"epoch": epoch, "train_loss": total / len(train_x), "val_acc": val_acc})
        if val_acc > best_acc:
            best_acc, best_state, bad_epochs = val_acc, copy.deepcopy(head.state_dict()), 0
        else:
            bad_epochs += 1
            if bad_epochs >= hc["patience"]:
                break
    head.load_state_dict(best_state)
    return head.cpu().eval(), {"best_val_acc": best_acc, "epochs_run": len(log), "log": log}


if (CACHE / "heads.pt").exists() and (CACHE / "clip_text.pt").exists():
    head_states = torch.load(CACHE / "heads.pt")
else:
    # features of the 5000 training images, then one head per backbone
    train_raw, train_y, _ = load_stl10("train")
    train_imgs = resize_to_common(train_raw)
    y, tr, va = torch.from_numpy(train_y), SPLITS["train_idx"], SPLITS["val_idx"]
    head_states, head_logs = {}, {}
    for bb in BACKBONES:
        feats = extract_features(get_backbone(bb), train_imgs)
        head, log = train_head(feats[tr], y[tr], feats[va], y[va], len(CLASS_NAMES))
        head_states[f"{bb}_head"], head_logs[f"{bb}_head"] = head.state_dict(), log
        print(f"{bb}: feature dim {feats.shape[1]}, best val acc {log['best_val_acc']:.4f}, "
              f"{log['epochs_run']} epochs")
    torch.save(head_states, CACHE / "heads.pt")
    save_json(head_logs, RESULTS / "head_training.json")

    # text side of CLIP for zero-shot, with the fixed prompt
    clip = get_backbone("clip")
    tokens = open_clip.get_tokenizer("ViT-B-32")([CFG["clip_prompt"].format(c) for c in CLASS_NAMES]).to(DEVICE)
    with torch.no_grad():
        text = F.normalize(clip.net.encode_text(tokens), dim=-1).float().cpu()
    torch.save({"text_feats": text, "logit_scale": float(clip.net.logit_scale.exp())}, CACHE / "clip_text.pt")
    del train_raw, train_imgs

HEADS = {}
for name, state in head_states.items():
    HEADS[name] = nn.Linear(state["weight"].shape[1], state["weight"].shape[0])
    HEADS[name].load_state_dict(state)
    HEADS[name].eval()
CLIP_TEXT = torch.load(CACHE / "clip_text.pt")

In [ ]:
CLASSIFIERS = ["resnet50_head", "vit_b16_head", "clip_head", "clip_zeroshot"]
FEATURE_OF = {"resnet50_head": "resnet50", "vit_b16_head": "vit_b16",
              "clip_head": "clip", "clip_zeroshot": "clip"}     # which backbone feeds which classifier


@torch.no_grad()
def predict(feats):
    """feats: {backbone: (N,D)} -> {classifier: (predicted class, confidence)}"""
    out = {}
    for clf in CLASSIFIERS:
        x = feats[FEATURE_OF[clf]]
        if clf == "clip_zeroshot":
            logits = CLIP_TEXT["logit_scale"] * x @ CLIP_TEXT["text_feats"].T   # scaled similarity to each prompt
        else:
            logits = HEADS[clf](x)
        conf, pred = logits.softmax(dim=1).max(dim=1)
        out[clf] = (pred.numpy(), conf.numpy())
    return out


def accuracy(pred, y):
    return float((pred == y).mean())


def consistency(pred_clean, pred_changed):
    """Share of images whose predicted class did not change."""
    return float((pred_clean == pred_changed).mean())


def intervention_table(conditions):
    """Accuracy, change vs clean and prediction consistency for the given conditions."""
    rows = []
    for clf in CLASSIFIERS:
        clean_pred, clean_conf = PREDS["clean"][clf]
        clean_acc = accuracy(clean_pred, LABELS)
        rows.append({"classifier": clf, "condition": "clean", "acc": clean_acc, "acc_change": 0.0,
                     "consistency": 1.0, "mean_max_conf": float(clean_conf.mean())})
        for cond in conditions:
            pred, conf = PREDS[cond][clf]
            rows.append({"classifier": clf, "condition": cond, "acc": accuracy(pred, LABELS),
                         "acc_change": accuracy(pred, LABELS) - clean_acc,
                         "consistency": consistency(clean_pred, pred),
                         "mean_max_conf": float(conf.mean())})
    return pd.DataFrame(rows)


FEATS, PREDS = {}, {}     # filled step by step below, keyed by condition name

## Step 1: Clean baseline

In [ ]:
FEATS["clean"] = features_for("clean", CLEAN)
PREDS["clean"] = predict(FEATS["clean"])

rows = []
for clf in CLASSIFIERS:
    pred, conf = PREDS["clean"][clf]
    rows.append({"classifier": clf, "top1_acc": accuracy(pred, LABELS),
                 "macro_f1": float(f1_score(LABELS, pred, average="macro")),
                 "mean_max_conf": float(conf.mean())})
save_table(pd.DataFrame(rows), "table_step1_clean_baseline")

## Step 2: Color bias

In [ ]:
# grayscale removes colour, hue rotation makes it wrong (shifting every color to its opposite). geometry is same in both.
GRAY = grayscale(CLEAN)
HUE = hue_rotate(CLEAN, CFG["hue_degrees"])

FEATS["gray"] = features_for("gray", GRAY)
FEATS["hue"] = features_for(f"hue{CFG['hue_degrees']}", HUE)
PREDS["gray"], PREDS["hue"] = predict(FEATS["gray"]), predict(FEATS["hue"])

save_table(intervention_table(["gray", "hue"]), "table_step2_color")

## Step 3: Shape vs. texture
### 3a. AdaIN and candidate images

In [ ]:
# AdaIN style transfer (Huang & Belongie, 2017). Layer layout, adain function and pretrained
# weights are from https://github.com/naoto0804/pytorch-AdaIN (MIT license), inference part only.
ADAIN_URL = "https://github.com/naoto0804/pytorch-AdaIN/releases/download/v0.0.0/"


def _conv(cin, cout):
    return [nn.ReflectionPad2d(1), nn.Conv2d(cin, cout, 3), nn.ReLU()]


def _pool():
    return nn.MaxPool2d(2, 2, 0, ceil_mode=True)


def build_encoder():
    """VGG-19 up to relu4_1. Layer order has to match the released weights."""
    layers = [nn.Conv2d(3, 3, 1)]
    layers += _conv(3, 64) + _conv(64, 64) + [_pool()]
    layers += _conv(64, 128) + _conv(128, 128) + [_pool()]
    layers += _conv(128, 256) + _conv(256, 256) + _conv(256, 256) + _conv(256, 256) + [_pool()]
    layers += _conv(256, 512)
    return nn.Sequential(*layers)


def build_decoder():
    up = lambda: nn.Upsample(scale_factor=2, mode="nearest")
    layers = _conv(512, 256) + [up()]
    layers += _conv(256, 256) + _conv(256, 256) + _conv(256, 256) + _conv(256, 128) + [up()]
    layers += _conv(128, 128) + _conv(128, 64) + [up()]
    layers += _conv(64, 64) + [nn.ReflectionPad2d(1), nn.Conv2d(64, 3, 3)]
    return nn.Sequential(*layers)


def _mean_std(feat, eps=1e-5):
    n, c = feat.shape[:2]
    flat = feat.reshape(n, c, -1)
    return flat.mean(dim=2).view(n, c, 1, 1), (flat.var(dim=2) + eps).sqrt().view(n, c, 1, 1)


def adain(content_feat, style_feat):
    """Give the content features the channel-wise mean/std of the style features."""
    c_mean, c_std = _mean_std(content_feat)
    s_mean, s_std = _mean_std(style_feat)
    return (content_feat - c_mean) / c_std * s_std + s_mean


class AdaINStylizer(nn.Module):
    def __init__(self, weights_dir):
        super().__init__()
        weights_dir.mkdir(parents=True, exist_ok=True)
        for fname in ["vgg_normalised.pth", "decoder.pth"]:            # fetched once
            if not (weights_dir / fname).exists():
                torch.hub.download_url_to_file(ADAIN_URL + fname, str(weights_dir / fname))

        # the vgg file holds the whole network, only the layers up to relu4_1 are needed
        self.encoder = build_encoder()
        vgg_state = torch.load(weights_dir / "vgg_normalised.pth", map_location="cpu")
        vgg_state = {k: v for k, v in vgg_state.items() if int(k.split(".")[0]) < len(self.encoder)}
        self.encoder.load_state_dict(vgg_state)
        self.decoder = build_decoder()
        self.decoder.load_state_dict(torch.load(weights_dir / "decoder.pth", map_location="cpu"))
        self.eval()
        for p in self.parameters():
            p.requires_grad_(False)

    @torch.no_grad()
    def forward(self, content_uint8, style_uint8, alpha=1.0):
        """content gives the layout, style gives the texture."""
        c_feat = self.encoder(content_uint8.float() / 255)
        s_feat = self.encoder(style_uint8.float() / 255)
        mixed = adain(c_feat, s_feat)
        mixed = alpha * mixed + (1 - alpha) * c_feat       # alpha < 1 keeps part of the content statistics
        return to_uint8(self.decoder(mixed))

In [ ]:
CUE_DIR = RESULTS / "cue_conflict"


def cue_groups():
    """Every pair in both directions: (shape class, texture class)."""
    groups = []
    for a, b in CFG["cue"]["pairs"]:
        groups += [(CLASS_NAMES.index(a), CLASS_NAMES.index(b)), (CLASS_NAMES.index(b), CLASS_NAMES.index(a))]
    return groups


def generate_candidates():
    n_per_group = CFG["cue"]["candidates_per_direction"]
    stylizer = AdaINStylizer(CACHE / "adain_weights").to(DEVICE)
    manifest, images = [], []
    for g, (shape_c, tex_c) in enumerate(cue_groups()):
        # each group has its own generator, so asking for more candidates later
        # never changes which images the other groups get
        rng = np.random.default_rng([CFG["seed"], g])
        content_pos = rng.permutation(np.where(LABELS == shape_c)[0])[:n_per_group]
        style_pos = rng.permutation(np.where(LABELS == tex_c)[0])[:n_per_group]

        for start in range(0, n_per_group, 10):
            c = CLEAN[content_pos[start:start + 10]].to(DEVICE)
            s = CLEAN[style_pos[start:start + 10]].to(DEVICE)
            images.append(stylizer(c, s, alpha=CFG["cue"]["alpha"]).cpu())

        for k in range(n_per_group):
            manifest.append({"id": g * 1000 + k,                  # 3012 = group 3, candidate 12
                             "group": g,
                             "shape_class": CLASS_NAMES[shape_c], "texture_class": CLASS_NAMES[tex_c],
                             "shape_label": int(shape_c), "texture_label": int(tex_c),
                             "content_pos": int(content_pos[k]),   # position inside the 500-image subset
                             "style_pos": int(style_pos[k])})
    return torch.cat(images), manifest


def save_contact_sheets(images, manifest, cols=5):
    """One sheet per group, each tile is content | stylized | style with its id on top."""
    to_np = lambda t: t.permute(1, 2, 0).numpy()
    gap = np.full((CFG["image_size"], 6, 3), 255, dtype=np.uint8)
    for g in sorted({m["group"] for m in manifest}):
        members = [i for i, m in enumerate(manifest) if m["group"] == g]
        n_rows = int(np.ceil(len(members) / cols))
        fig, axes = plt.subplots(n_rows, cols, figsize=(cols * 4.2, n_rows * 1.75))
        axes = np.atleast_2d(axes)
        for ax in axes.ravel():
            ax.axis("off")
        for ax, i in zip(axes.ravel(), members):
            m = manifest[i]
            ax.imshow(np.concatenate([to_np(CLEAN[m["content_pos"]]), gap, to_np(images[i]), gap,
                                      to_np(CLEAN[m["style_pos"]])], axis=1))
            ax.set_title(f"id {m['id']}", fontsize=10, pad=2)
        first = manifest[members[0]]
        fig.suptitle(f"group {g}:  shape = {first['shape_class']}   |   texture = {first['texture_class']}"
                     "      (each tile: content | stylized | style)", fontsize=13)
        fig.tight_layout(rect=(0, 0, 1, 0.97))
        fig.savefig(CUE_DIR / "sheets" / f"group_{g}_{first['shape_class']}_as_{first['texture_class']}.jpg",
                    dpi=100, pil_kwargs={"quality": 85})
        plt.close(fig)


if (CACHE / "cue_candidates.pt").exists() and (CUE_DIR / "candidates_manifest.json").exists():
    print("candidates already generated, keeping them")
else:
    cand_images, cand_manifest = generate_candidates()
    torch.save(cand_images, CACHE / "cue_candidates.pt")
    save_json({"alpha": CFG["cue"]["alpha"], "rejection_rule": CFG["cue"]["rejection_rule"],
               "candidates": cand_manifest}, CUE_DIR / "candidates_manifest.json")
    save_contact_sheets(cand_images, cand_manifest)
    print(f"{len(cand_manifest)} candidates generated")

### 3b. Review by eye
Rejection rule (fixed before looking at any model output): (`CFG["cue"]["rejection_rule"]`.)

In [ ]:
from IPython.display import Image, display
print(CFG["cue"]["rejection_rule"])
for sheet in sorted((CUE_DIR / "sheets").glob("*.jpg")):
    display(Image(filename=str(sheet)))

In [ ]:
# ids of the candidates that break the rejection rule
rejected = [0,2,3,4,17,18,21,28,1004,1012,1015,1020,1024,2005,2006,2007,2024,3000, 3001, 3006, 3010, 3011, 3012, 3016, 3023, 4002, 4003, 4005, 4016, 4020, 4021, 4024, 4025, 4027,4028,5007, 5008, 5010, 5014, 5015, 5016, 5018, 5020, 5027, 5028, 6004, 6017, 6019, 6020, 6021, 6022, 6024, 7002, 7017, 7018, 7028, 9008, 9021, 9022]

save_json(sorted(set(rejected)), CUE_DIR / "rejected_ids.json")

# apply the rejections, then keep the same number of images in every group
cand = load_json(CUE_DIR / "candidates_manifest.json")["candidates"]
unknown = set(rejected) - {m["id"] for m in cand}
assert not unknown, f"these ids do not exist: {sorted(unknown)}"

groups = sorted({m["group"] for m in cand})
survivors = {g: [i for i, m in enumerate(cand) if m["group"] == g and m["id"] not in set(rejected)]
             for g in groups}
keep_n = min(len(v) for v in survivors.values())            # size of the smallest group
keep = [i for g in groups for i in survivors[g][:keep_n]]

CUE = torch.load(CACHE / "cue_candidates.pt")[keep].clone()
CUE_MANIFEST = [cand[i] for i in keep]
save_json({"rejection_rule": CFG["cue"]["rejection_rule"], "alpha": CFG["cue"]["alpha"],
           "n_candidates": len(cand), "n_rejected_by_eye": len(set(rejected)),
           "n_passed_review": len(cand) - len(set(rejected)),
           "n_dropped_for_balance": len(cand) - len(set(rejected)) - len(keep),
           "n_final": len(keep), "per_group_final": keep_n,
           "per_group_passed_review": {str(g): len(v) for g, v in survivors.items()},
           "images": CUE_MANIFEST}, CUE_DIR / "accepted_manifest.json")

# features from an older accepted set would be stale
(CACHE / "feats" / "cue.pt").unlink(missing_ok=True)

print(f"candidates {len(cand)} | rejected by eye {len(set(rejected))} | final set {len(keep)} ({keep_n} per group)")
if len(keep) < CFG["cue"]["min_total"]:
    print("fewer than the required total: raise candidates_per_direction, delete cue_candidates.pt "
          "from the cache folder, rerun 3a and review the new ids")

### 3c. Shape, texture or other

In [ ]:
FEATS["cue"] = features_for("cue", CUE)
PREDS["cue"] = predict(FEATS["cue"])

SHAPE_Y = np.array([m["shape_label"] for m in CUE_MANIFEST])
TEX_Y = np.array([m["texture_label"] for m in CUE_MANIFEST])
CONTENT_POS = [m["content_pos"] for m in CUE_MANIFEST]       # clean counterpart of each cue-conflict image
PAIR_OF = np.array([m["group"] // 2 for m in CUE_MANIFEST])  # groups 2k and 2k+1 are the same pair


def decision_types(pred):
    """'shape', 'texture' or 'other' for every cue-conflict image."""
    return np.where(pred == SHAPE_Y, "shape", np.where(pred == TEX_Y, "texture", "other"))


def summarize_decisions(kinds):
    n_s, n_t, n_o = [(kinds == k).sum() for k in ("shape", "texture", "other")]
    return {"n_shape": int(n_s), "n_texture": int(n_t), "n_other": int(n_o), "n_total": int(len(kinds)),
            "shape_bias_pct": float(100 * n_s / max(n_s + n_t, 1)),
            "coverage_pct": float(100 * (n_s + n_t) / len(kinds))}


overall, per_pair = [], []
for clf in CLASSIFIERS:
    kinds = decision_types(PREDS["cue"][clf][0])
    overall.append({"classifier": clf, **summarize_decisions(kinds)})
    for p in np.unique(PAIR_OF):
        a, b = CFG["cue"]["pairs"][p]
        per_pair.append({"classifier": clf, "pair": f"{a}-{b}", **summarize_decisions(kinds[PAIR_OF == p])})
save_table(pd.DataFrame(overall), "table_step3_shape_bias")
save_table(pd.DataFrame(per_pair), "table_step3_shape_bias_per_pair")

# did the prediction survive stylization at all? (compared with the content image)
CUE_CONSISTENCY = {clf: consistency(PREDS["clean"][clf][0][CONTENT_POS], PREDS["cue"][clf][0])
                   for clf in CLASSIFIERS}
save_json(CUE_CONSISTENCY, RESULTS / "cue_conflict_prediction_consistency.json")

## Step 4: Translation

In [ ]:
# every shift size in all four directions, each set built once and shown to all models
for d in CFG["translation_shifts"]:
    if d == 0:
        continue                    # a shift of 0 is the clean image
    for direction, (sx, sy) in DIRECTIONS.items():
        key = f"trans_{d}_{direction}"
        FEATS[key] = features_for(key, translate(CLEAN, dx=sx * d, dy=sy * d))
        PREDS[key] = predict(FEATS[key])

rows, rows_by_dir = [], []
for clf in CLASSIFIERS:
    clean_pred = PREDS["clean"][clf][0]
    for d in CFG["translation_shifts"]:
        if d == 0:
            rows.append({"classifier": clf, "shift_px": 0, "acc": accuracy(clean_pred, LABELS), "consistency": 1.0})
            continue
        accs, cons = [], []
        for direction in DIRECTIONS:
            pred = PREDS[f"trans_{d}_{direction}"][clf][0]
            accs.append(accuracy(pred, LABELS))
            cons.append(consistency(clean_pred, pred))
            rows_by_dir.append({"classifier": clf, "shift_px": d, "direction": direction,
                                "acc": accs[-1], "consistency": cons[-1]})
        # average over the four directions
        rows.append({"classifier": clf, "shift_px": d, "acc": float(np.mean(accs)),
                     "consistency": float(np.mean(cons))})
TRANS_TABLE = pd.DataFrame(rows)
save_table(TRANS_TABLE, "table_step4_translation")
pd.DataFrame(rows_by_dir).to_csv(RESULTS / "table_step4_translation_by_direction.csv", index=False)

## Step 5: Patch structure

In [ ]:
# 4x4 grid, one random non-identity order per image, the same shuffled images for every model
SHUFFLED, PERMS = patch_shuffle(CLEAN, CFG["patch_grid"], CFG["seed"])
save_json({"grid": CFG["patch_grid"], "seed": CFG["seed"], "permutations": PERMS},
          RESULTS / "patch_permutations.json")

FEATS["patch"] = features_for("patch", SHUFFLED)
PREDS["patch"] = predict(FEATS["patch"])
save_table(intervention_table(["patch"]), "table_step5_patch_shuffle")

In [ ]:
# compact side-by-side view: clean, grayscale, hue rotation, patch shuffle
COMPACT = intervention_table(["gray", "hue", "patch"])
save_table(COMPACT, "table_compact_clean_color_patch")

# how often does CLIP zero-shot agree with the CLIP head?
save_table(pd.DataFrame([{"condition": cond, "zeroshot_vs_head_agreement":
                          consistency(p["clip_head"][0], p["clip_zeroshot"][0])} for cond, p in PREDS.items()]),
           "table_clip_zeroshot_vs_head")

# every single prediction, so any number above can be traced back
save_json({cond: {clf: {"pred": p.tolist(), "conf": np.round(c, 4).tolist()} for clf, (p, c) in d.items()}
           for cond, d in PREDS.items()}, RESULTS / "predictions.json")

## Step 6: Representation analysis
### 6a. Cosine stability

In [ ]:
def cosine_stability(clean_feats, changed_feats):
    """Mean cosine similarity between each clean image and its transformed version."""
    return float(F.cosine_similarity(clean_feats, changed_feats, dim=1).mean())


rows = []
for bb in BACKBONES:
    clean = FEATS["clean"][bb]
    rows.append({"backbone": bb, "intervention": "grayscale", "cosine_stability": cosine_stability(clean, FEATS["gray"][bb])})
    rows.append({"backbone": bb, "intervention": "hue_rotation", "cosine_stability": cosine_stability(clean, FEATS["hue"][bb])})
    # a cue-conflict image is compared with the content image it was made from
    rows.append({"backbone": bb, "intervention": "cue_conflict",
                 "cosine_stability": cosine_stability(clean[CONTENT_POS], FEATS["cue"][bb])})
    for d in CFG["translation_shifts"]:
        if d == 0:
            continue
        per_dir = [cosine_stability(clean, FEATS[f"trans_{d}_{direction}"][bb]) for direction in DIRECTIONS]
        rows.append({"backbone": bb, "intervention": f"translation_{d}px", "cosine_stability": float(np.mean(per_dir))})
    rows.append({"backbone": bb, "intervention": "patch_shuffle", "cosine_stability": cosine_stability(clean, FEATS["patch"][bb])})
STAB_TABLE = pd.DataFrame(rows)
save_table(STAB_TABLE.pivot(index="backbone", columns="intervention", values="cosine_stability")
           .loc[BACKBONES].reset_index(), "table_step6_cosine_stability")

### 6b. 2-D maps of clean and transformed features

In [ ]:
SHOWN = ["gray", "cue", "trans", "patch"]      # conditions drawn next to the clean points


def project_backbone(bb):
    """ONE fit per backbone on clean + all transformed features together,
    so every condition lives in the same 2-D space."""
    pc = CFG["projection"]
    blocks = {"clean": FEATS["clean"][bb], "gray": FEATS["gray"][bb], "cue": FEATS["cue"][bb],
              "trans": FEATS[f"trans_{pc['translation_shown']}_right"][bb], "patch": FEATS["patch"][bb]}
    stacked = np.concatenate([b.numpy() for b in blocks.values()])

    if pc["method"] == "tsne":
        reducer = TSNE(n_components=2, perplexity=pc["perplexity"], metric=pc["metric"], init="pca",
                       learning_rate="auto", random_state=CFG["seed"])
    else:
        import umap
        reducer = umap.UMAP(n_components=2, n_neighbors=pc["n_neighbors"], min_dist=pc["min_dist"],
                            metric=pc["metric"], random_state=CFG["seed"])
    xy = reducer.fit_transform(stacked)

    # cut the big array back into one piece per condition
    out, start = {}, 0
    for name, block in blocks.items():
        # cue-conflict points get the class of their content (shape) image
        out[name] = {"xy": xy[start:start + len(block)], "labels": LABELS[CONTENT_POS] if name == "cue" else LABELS}
        start += len(block)
    return out


PROJ = {bb: project_backbone(bb) for bb in BACKBONES}
save_json(CFG["projection"], RESULTS / "projection_settings.json")

## Figures

In [ ]:
def show_img(t):
    return t.permute(1, 2, 0).numpy()


def bar_style(clf):
    # zero-shot CLIP has no trained head, the hatch sets it apart
    return {"color": MODEL_COLORS[clf], "edgecolor": "white", "linewidth": 0.6,
            "hatch": "///" if clf == "clip_zeroshot" else None}


# one image under every intervention (the content image of the first cue conflict)
pos, shift = CONTENT_POS[0], CFG["projection"]["translation_shown"]
panels = {"Clean": CLEAN[pos], "Grayscale": GRAY[pos], f"Hue +{CFG['hue_degrees']}°": HUE[pos],
          f"Shift {shift}px": translate(CLEAN[pos:pos + 1], dx=shift)[0], "Patch shuffle": SHUFFLED[pos],
          f"Cue conflict\n({CUE_MANIFEST[0]['texture_class']} texture)": CUE[0]}
fig, axes = plt.subplots(1, len(panels), figsize=(1.55 * len(panels), 1.85))
for ax, (title, im) in zip(axes, panels.items()):
    ax.imshow(show_img(im))
    ax.set_title(title, fontsize=8)
    ax.axis("off")
fig.tight_layout(w_pad=0.4)
save_fig(fig, "fig_interventions_overview")

In [ ]:
# clean vs colour vs patch shuffle: accuracy on the left, unchanged predictions on the right
COND_NAMES = {"clean": "Clean", "gray": "Grayscale", "hue": "Hue\nrotation", "patch": "Patch\nshuffle"}
conds, width = ["clean", "gray", "hue", "patch"], 0.2
fig, (ax_acc, ax_con) = plt.subplots(1, 2, figsize=(7.0, 2.7), gridspec_kw={"width_ratios": [4, 3]})
for j, clf in enumerate(CLASSIFIERS):
    sub = COMPACT[COMPACT.classifier == clf].set_index("condition")
    offset = (j - 1.5) * width
    for ax, names, col in [(ax_acc, conds, "acc"), (ax_con, conds[1:], "consistency")]:
        vals = [100 * sub.loc[c, col] for c in names]
        bars = ax.bar(np.arange(len(names)) + offset, vals, width, label=MODEL_LABELS[clf], **bar_style(clf))
        for b, v in zip(bars, vals):
            ax.text(b.get_x() + b.get_width() / 2, v + 1, f"{v:.0f}", ha="center", fontsize=6, color=INK)
for ax, names, title in [(ax_acc, conds, "Top-1 accuracy (%)"), (ax_con, conds[1:], "Prediction consistency vs clean (%)")]:
    ax.set_xticks(np.arange(len(names)))
    ax.set_xticklabels([COND_NAMES[c] for c in names])
    ax.set_ylim(0, 112)
    ax.set_yticks([0, 25, 50, 75, 100])
    ax.set_title(title)
    ax.grid(axis="x", visible=False)
fig.legend(*ax_acc.get_legend_handles_labels(), loc="lower center", ncol=4, bbox_to_anchor=(0.5, -0.09))
fig.tight_layout()
save_fig(fig, "fig_color_patch_performance")

In [ ]:
# shape / texture / other decisions, with shape bias and coverage written next to each bar
table = pd.DataFrame(overall).set_index("classifier")
fig, ax = plt.subplots(figsize=(7.0, 2.1))
y = np.arange(len(CLASSIFIERS))[::-1]
left = np.zeros(len(CLASSIFIERS))
for kind in ["shape", "texture", "other"]:
    share = np.array([100 * table.loc[c, f"n_{kind}"] / table.loc[c, "n_total"] for c in CLASSIFIERS])
    ax.barh(y, share, left=left, color=DECISION_COLORS[kind], edgecolor="white", height=0.62, label=f"{kind} decision")
    for yi, l, s, c in zip(y, left, share, CLASSIFIERS):
        if s >= 7:      # only label segments wide enough to hold the number
            ax.text(l + s / 2, yi, f"{int(table.loc[c, f'n_{kind}'])}", ha="center", va="center",
                    fontsize=7.5, color="white" if kind != "other" else INK)
    left += share
for yi, c in zip(y, CLASSIFIERS):
    ax.text(102, yi, f"shape bias {table.loc[c, 'shape_bias_pct']:.1f}%  |  coverage {table.loc[c, 'coverage_pct']:.1f}%",
            va="center", fontsize=7.5, color=INK)
ax.set_yticks(y)
ax.set_yticklabels([MODEL_LABELS[c] for c in CLASSIFIERS])
ax.set_xlim(0, 100)
ax.set_xlabel("share of cue-conflict images (%), counts inside bars")
ax.grid(axis="y", visible=False)
ax.legend(loc="lower center", ncol=3, bbox_to_anchor=(0.5, 1.0))
save_fig(fig, "fig_shape_texture_decisions")

In [ ]:
# translation curves
fig, axes = plt.subplots(1, 2, figsize=(7.0, 2.5))
for clf in CLASSIFIERS:
    sub = TRANS_TABLE[TRANS_TABLE.classifier == clf].sort_values("shift_px")
    for ax, col in zip(axes, ["acc", "consistency"]):
        ax.plot(sub.shift_px, 100 * sub[col], color=MODEL_COLORS[clf], marker=MODEL_MARKERS[clf], markersize=5,
                linewidth=1.8, markeredgecolor="white", markeredgewidth=0.6,
                linestyle="--" if clf == "clip_zeroshot" else "-", label=MODEL_LABELS[clf])
lowest = 100 * min(TRANS_TABLE.acc.min(), TRANS_TABLE.consistency.min())
for ax, title in zip(axes, ["Top-1 accuracy (%)", "Prediction consistency vs unshifted (%)"]):
    ax.set_xticks(CFG["translation_shifts"])
    ax.set_xlabel("shift (pixels), mean of 4 directions")
    ax.set_title(title)
    ax.set_ylim(max(0, np.floor(lowest / 5) * 5 - 5), 101)     # zoom in on the range the curves cover
fig.legend(*axes[0].get_legend_handles_labels(), loc="lower center", ncol=4, bbox_to_anchor=(0.5, -0.1))
fig.tight_layout()
save_fig(fig, "fig_translation_curves")

In [ ]:
# feature stability heatmap
NICE = {"grayscale": "Grayscale", "hue_rotation": "Hue\nrotation", "cue_conflict": "Cue\nconflict",
        "translation_8px": "Shift\n8px", "translation_16px": "Shift\n16px", "translation_32px": "Shift\n32px",
        "patch_shuffle": "Patch\nshuffle"}
order = [i for i in NICE if i in set(STAB_TABLE.intervention)]
grid = STAB_TABLE.pivot(index="backbone", columns="intervention", values="cosine_stability").loc[BACKBONES, order]
vmin = min(0.0, float(grid.values.min()))

fig, ax = plt.subplots(figsize=(7.0, 1.9))
im = ax.imshow(grid.values, cmap=STABILITY_CMAP, vmin=vmin, vmax=1, aspect="auto")
for r in range(grid.shape[0]):
    for c in range(grid.shape[1]):
        v = grid.values[r, c]
        ax.text(c, r, f"{v:.2f}", ha="center", va="center", fontsize=8.5,
                color="white" if (v - vmin) / (1 - vmin) > 0.55 else INK)
ax.set_xticks(range(len(order)))
ax.set_xticklabels([NICE[i] for i in order])
ax.set_yticks(range(len(BACKBONES)))
ax.set_yticklabels([BACKBONE_LABELS[b] for b in BACKBONES])
ax.grid(False)
for s in ax.spines.values():
    s.set_visible(False)
cb = fig.colorbar(im, ax=ax, pad=0.015)
cb.set_label("cosine to clean feature", fontsize=8)
cb.outline.set_visible(False)
save_fig(fig, "fig_feature_stability")

In [ ]:
# did predictions change as much as the features did?
biggest = max(CFG["translation_shifts"])
marker_of = {"grayscale": "o", "hue_rotation": "P", f"translation_{biggest}px": "s", "patch_shuffle": "X", "cue_conflict": "*"}
compact_name = {"grayscale": "gray", "hue_rotation": "hue", "patch_shuffle": "patch"}

fig, ax = plt.subplots(figsize=(3.6, 3.0))
for clf in CLASSIFIERS:
    for inter, mk in marker_of.items():
        x = STAB_TABLE[(STAB_TABLE.backbone == FEATURE_OF[clf]) & (STAB_TABLE.intervention == inter)].cosine_stability.item()
        if inter in compact_name:
            y_val = COMPACT[(COMPACT.classifier == clf) & (COMPACT.condition == compact_name[inter])].consistency.item()
        elif inter == "cue_conflict":
            y_val = CUE_CONSISTENCY[clf]
        else:
            y_val = TRANS_TABLE[(TRANS_TABLE.classifier == clf) & (TRANS_TABLE.shift_px == biggest)].consistency.item()
        ax.scatter(x, 100 * y_val, marker=mk, s=120 if mk == "*" else 70, linewidth=1.4, zorder=3,
                   facecolor="none" if clf == "clip_zeroshot" else MODEL_COLORS[clf], edgecolor=MODEL_COLORS[clf])
ax.set_xlabel("feature stability (cosine to clean)")
ax.set_ylabel("prediction consistency (%)")
h1 = [Line2D([], [], marker="o", linestyle="", markeredgewidth=1.4, markeredgecolor=MODEL_COLORS[c],
             markerfacecolor="none" if c == "clip_zeroshot" else MODEL_COLORS[c], label=MODEL_LABELS[c]) for c in CLASSIFIERS]
h2 = [Line2D([], [], marker=m, linestyle="", color="#777777", label=NICE[i].replace("\n", " "))
      for i, m in marker_of.items()]
# figure-level legends, so nothing gets cut off when saving
fig.legend(handles=h1, loc="upper left", bbox_to_anchor=(0.92, 0.9), title="classifier", title_fontsize=8)
fig.legend(handles=h2, loc="lower left", bbox_to_anchor=(0.92, 0.1), title="intervention", title_fontsize=8)
save_fig(fig, "fig_prediction_vs_feature_change")

In [ ]:
# t-SNE / UMAP maps: colour = true class, circle = clean, triangle = transformed
def draw_map(ax, proj, cond):
    colors = np.array(CLASS_COLORS)
    ax.scatter(*proj["clean"]["xy"].T, c=colors[proj["clean"]["labels"]], s=9, marker="o", alpha=0.35, linewidths=0)
    ax.scatter(*proj[cond]["xy"].T, c=colors[proj[cond]["labels"]], s=13, marker="^", alpha=0.95,
               edgecolors=INK, linewidths=0.35)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.grid(False)
    for s in ax.spines.values():
        s.set_visible(True)
        s.set_color("#BDB8AC")


def map_legend(fig):
    handles = [Line2D([], [], marker="o", linestyle="", color=CLASS_COLORS[i], markersize=5, label=n)
               for i, n in enumerate(CLASS_NAMES)]
    handles += [Line2D([], [], marker="o", linestyle="", color="#999999", alpha=0.5, markersize=5, label="clean"),
                Line2D([], [], marker="^", linestyle="", markerfacecolor="#999999", markeredgecolor=INK,
                       markersize=6, label="transformed")]
    fig.legend(handles=handles, loc="lower center", ncol=6, bbox_to_anchor=(0.5, 0.0), columnspacing=1.2, handletextpad=0.2)


method = "t-SNE" if CFG["projection"]["method"] == "tsne" else "UMAP"
titles = {"gray": "Grayscale", "cue": "Cue conflict\n(colour = shape class)",
          "trans": f"Shift {CFG['projection']['translation_shown']}px right", "patch": "Patch shuffle"}

# full grid: rows = backbones, columns = interventions, one fit per row
fig, axes = plt.subplots(len(BACKBONES), len(SHOWN), figsize=(7.0, 5.6))
for r, bb in enumerate(BACKBONES):
    for c, cond in enumerate(SHOWN):
        draw_map(axes[r, c], PROJ[bb], cond)
        if r == 0:
            axes[r, c].set_title(titles[cond], fontsize=8)
    axes[r, 0].set_ylabel(BACKBONE_LABELS[bb], fontsize=9, color=BACKBONE_COLORS[bb], fontweight="bold")
fig.suptitle(f"{method} of clean and transformed features (one fit per backbone)", fontsize=10, y=0.995)
fig.tight_layout(rect=(0, 0.07, 1, 1), w_pad=0.3, h_pad=0.3)
map_legend(fig)
save_fig(fig, "fig_projection_all")

# one-row versions, for when space in the report is tight
for cond in SHOWN:
    fig, axes = plt.subplots(1, len(BACKBONES), figsize=(7.0, 2.6))
    for ax, bb in zip(axes, BACKBONES):
        draw_map(ax, PROJ[bb], cond)
        ax.set_title(BACKBONE_LABELS[bb], fontsize=9, color=BACKBONE_COLORS[bb], fontweight="bold")
    fig.tight_layout(rect=(0, 0.13, 1, 1), w_pad=0.3)
    map_legend(fig)
    save_fig(fig, f"fig_projection_{cond}")

In [ ]:
# cue-conflict examples: a few images of each kind, with every model's answer underneath
def annotate(ax, lines):
    for k, (text, color) in enumerate(lines):
        ax.text(0.0, -0.06 - 0.115 * k, text, transform=ax.transAxes, fontsize=6.6, color=color,
                va="top", ha="left", fontweight="bold")


KINDS = np.stack([decision_types(PREDS["cue"][c][0]) for c in CLASSIFIERS])      # (4 classifiers, N images)
n_shape, n_tex, n_other = [(KINDS == k).sum(0) for k in ("shape", "texture", "other")]
buckets = {"all models follow shape": np.where(n_shape == 4)[0],
           "all models follow texture": np.where(n_tex == 4)[0],
           "models disagree": np.where((n_shape >= 1) & (n_tex >= 1))[0],
           "neither class predicted": np.where(n_other >= 2)[0]}
rng = np.random.default_rng(CFG["seed"])
picks = [(name, int(i)) for name, idx in buckets.items() for i in rng.permutation(idx)[:2]]

gap = np.full((CFG["image_size"], 8, 3), 255, dtype=np.uint8)
n_rows = int(np.ceil(len(picks) / 2))
fig, axes = plt.subplots(n_rows, 2, figsize=(7.0, 2.35 * n_rows))
axes = np.atleast_2d(axes)
for ax in axes.ravel():
    ax.axis("off")
records = []
for ax, (bucket, i) in zip(axes.ravel(), picks):
    m = CUE_MANIFEST[i]
    ax.imshow(np.concatenate([show_img(CLEAN[m["content_pos"]]), gap, show_img(CLEAN[m["style_pos"]]), gap,
                              show_img(CUE[i])], axis=1))
    ax.set_title(f"{bucket}\nshape: {m['shape_class']}  |  texture: {m['texture_class']}  |  result", fontsize=7.5)
    lines, rec = [], {"id": m["id"], "bucket": bucket, "shape": m["shape_class"], "texture": m["texture_class"]}
    for j, clf in enumerate(CLASSIFIERS):
        pred, conf, kind = PREDS["cue"][clf][0][i], PREDS["cue"][clf][1][i], str(KINDS[j, i])
        lines.append((f"{MODEL_LABELS[clf]}: {CLASS_NAMES[pred]} ({kind}, {conf:.2f})",
                      DECISION_COLORS[kind] if kind != "other" else "#8A8578"))
        rec[clf] = {"pred": CLASS_NAMES[pred], "kind": kind, "conf": float(conf)}
    annotate(ax, lines)
    records.append(rec)
fig.subplots_adjust(hspace=0.55, wspace=0.08)
save_fig(fig, "fig_cue_conflict_examples")
save_json(records, RESULTS / "cue_conflict_examples.json")

In [ ]:
# browse: up to 8 cue-conflict images of each kind, with their ids, to find ones worth showing
to_np = lambda t: t.permute(1, 2, 0).numpy()
KINDS = np.stack([decision_types(PREDS["cue"][c][0]) for c in CLASSIFIERS])      # (4 classifiers, N images)
n_shape, n_tex, n_other = [(KINDS == k).sum(0) for k in ("shape", "texture", "other")]
buckets = {"all models follow shape": np.where(n_shape == 4)[0],
           "all models follow texture": np.where(n_tex == 4)[0],
           "models disagree": np.where((n_shape >= 1) & (n_tex >= 1))[0],
           "neither class predicted": np.where(n_other >= 2)[0]}

for name, idx in buckets.items():
    print(f"{name}: {len(idx)} images")
    if len(idx) == 0:
        continue
    fig, axes = plt.subplots(1, 8, figsize=(16, 2.4))
    for ax in axes:
        ax.axis("off")
    for ax, i in zip(axes, idx[:8]):
        m = CUE_MANIFEST[i]
        ax.imshow(to_np(CUE[i]))
        ax.set_title(f"id {m['id']}\n{m['shape_class']} shape / {m['texture_class']} texture", fontsize=8)
    plt.show()
    plt.close(fig)

In [ ]:
# one example per kind, all in a single row. put an id from the browse cell above, None = first one of that kind
chosen = {"all models follow shape": 16,
          "all models follow texture": 2019,
          "models disagree": 10,
          "neither class predicted": 15}

pos_of = {m["id"]: i for i, m in enumerate(CUE_MANIFEST)}
picks = []
for name, idx in buckets.items():
    if chosen[name] is not None:
        assert pos_of[chosen[name]] in idx, f"id {chosen[name]} is not a '{name}' image"
        picks.append((name, pos_of[chosen[name]]))
    elif len(idx) > 0:
        picks.append((name, int(idx[0])))

SHORT = {"resnet50_head": "ResNet", "vit_b16_head": "ViT", "clip_head": "CLIP+head", "clip_zeroshot": "CLIP-ZS"}
size = CFG["image_size"]

fig, axes = plt.subplots(1, 4, figsize=(7.0, 2.45))
for ax in axes:
    ax.axis("off")
records = []
for ax, (name, i) in zip(axes, picks):
    m = CUE_MANIFEST[i]
    # small content (top) and style (bottom) thumbnails on the left, big stylized result on the right
    content_small = to_np(CLEAN[m["content_pos"]])[::2, ::2]
    style_small = to_np(CLEAN[m["style_pos"]])[::2, ::2]
    left = np.concatenate([content_small, style_small], axis=0)
    gap = np.full((size, 6, 3), 255, dtype=np.uint8)
    ax.imshow(np.concatenate([left, gap, to_np(CUE[i])], axis=1))

    # bold heading on top, plain line with the two classes under it
    ax.set_title(f"shape: {m['shape_class']} | texture: {m['texture_class']}", fontsize=7, pad=3)
    ax.annotate(name, xy=(0.5, 1), xycoords="axes fraction", xytext=(0, 13), textcoords="offset points",
                ha="center", va="bottom", fontsize=7, fontweight="bold")

    rec = {"id": m["id"], "kind": name, "shape": m["shape_class"], "texture": m["texture_class"]}
    # each model's answer underneath, coloured by what it followed
    for k, clf in enumerate(CLASSIFIERS):
        pred, conf, kind = PREDS["cue"][clf][0][i], PREDS["cue"][clf][1][i], str(KINDS[k, i])
        ax.text(0.0, -0.06 - 0.125 * k, f"{SHORT[clf]}: {CLASS_NAMES[pred]} {conf:.2f}", transform=ax.transAxes,
                fontsize=6.5, fontweight="bold", va="top", ha="left",
                color=DECISION_COLORS[kind] if kind != "other" else "#8A8578")
        rec[clf] = {"pred": CLASS_NAMES[pred], "kind": kind, "conf": float(conf)}
    records.append(rec)

fig.subplots_adjust(bottom=0.38, wspace=0.12)
fig.canvas.draw()                       # so the panel positions are final
box = axes[0].get_position()
# legend goes right under the prediction text, not at the bottom edge of the figure
fig.legend(handles=[Patch(color=DECISION_COLORS["shape"], label="follows shape"),
                    Patch(color=DECISION_COLORS["texture"], label="follows texture"),
                    Patch(color="#8A8578", label="neither")],
           loc="upper center", ncol=3, fontsize=7, bbox_to_anchor=(0.5, box.y0 - 0.60 * box.height))

save_fig(fig, "fig_cue_conflict_examples_row")
save_json(records, RESULTS / "cue_conflict_examples_row.json")

In [ ]:
# shuffled images on which the models are most sure of themselves
mean_conf = np.mean([PREDS["patch"][c][1] for c in CLASSIFIERS], axis=0)
top = np.argsort(-mean_conf)[:6]
fig, axes = plt.subplots(1, len(top), figsize=(7.0, 2.3))
for ax, i in zip(axes, top):
    ax.imshow(show_img(SHUFFLED[i]))
    ax.axis("off")
    ax.set_title(f"true: {CLASS_NAMES[LABELS[i]]}", fontsize=7.5)
    annotate(ax, [(f"{CLASS_NAMES[PREDS['patch'][c][0][i]]} {PREDS['patch'][c][1][i]:.2f}", MODEL_TEXT_COLORS[c])
                  for c in CLASSIFIERS])
fig.legend(handles=[Patch(color=MODEL_COLORS[c], label=MODEL_LABELS[c]) for c in CLASSIFIERS],
           loc="lower center", ncol=4, bbox_to_anchor=(0.5, -0.02))
fig.subplots_adjust(bottom=0.42, wspace=0.06)
save_fig(fig, "fig_patch_shuffle_confident")

## committing to GitHub

In [ ]:
from google.colab import userdata
token = userdata.get("GH_TOKEN")
%cd {REPO}
!git config user.name "mardyweb"
!git config user.email "maryamw17@outlook.com"
!git remote set-url origin https://{token}@github.com/mardyweb/atml-pa1.git
!git add task1
!git commit -m "Task 1: notebook, results and figures"
!git push